In [ ]:
import os, re, math, time, random, pickle, hashlib
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler

import rasterio
from tqdm.auto import tqdm

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)


# ----------------------------
# Scan filenames (old training format)
# im_id1_cl_id2_date1_date2_area.tif  (id1 territory, cl label, id2 field)
# ----------------------------
FNAME_RE_OLD = re.compile(
    r"^im_(?P<id1>-?\d+)_(?P<cl>[01])_(?P<id2>-?\d+)_(?P<dmg>\d{8})_(?P<ref>\d{8})_(?P<area>[-+]?\d*\.?\d+)\.tif$"
)

@dataclass(frozen=True)
class SampleMeta:
    path: str
    id1: int
    id2: int
    cl: int
    dmg: str
    ref: str
    area: float
    field_key: Tuple[int, int]  # (id1, id2)


def scan_folder(folder: str) -> List[SampleMeta]:
    out = []
    for fn in os.listdir(folder):
        if not fn.lower().endswith(".tif"):
            continue
        m = FNAME_RE_OLD.match(fn)
        if not m:
            continue
        id1 = int(m.group("id1"))
        id2 = int(m.group("id2"))
        out.append(SampleMeta(
            path=os.path.join(folder, fn),
            id1=id1,
            id2=id2,
            cl=int(m.group("cl")),
            dmg=m.group("dmg"),
            ref=m.group("ref"),
            area=float(m.group("area")),
            field_key=(id1, id2)
        ))
    if not out:
        raise RuntimeError(f"No matching tif found in {folder}")
    return out


# ----------------------------
# Field split with 20% val per group
# group A: fields ever damaged
# group B: other fields
# ----------------------------
def make_field_split(samples: List[SampleMeta], val_frac_each_group=0.20, seed=42):
    rng = random.Random(seed)
    field_to_idxs: Dict[Tuple[int,int], List[int]] = {}
    for i,s in enumerate(samples):
        field_to_idxs.setdefault(s.field_key, []).append(i)

    field_is_damaged = {
        fk: any(samples[j].cl == 1 for j in idxs)
        for fk, idxs in field_to_idxs.items()
    }
    damaged_fields = [fk for fk, d in field_is_damaged.items() if d]
    other_fields   = [fk for fk, d in field_is_damaged.items() if not d]

    rng.shuffle(damaged_fields)
    rng.shuffle(other_fields)

    n_val_d = int(round(val_frac_each_group * len(damaged_fields))) if damaged_fields else 0
    n_val_o = int(round(val_frac_each_group * len(other_fields))) if other_fields else 0
    n_val_d = max(1, n_val_d) if damaged_fields else 0
    n_val_o = max(1, n_val_o) if other_fields else 0

    val_fields = set(damaged_fields[:n_val_d] + other_fields[:n_val_o])
    train_fields = set(field_to_idxs.keys()) - val_fields

    train_idx = [i for fk in train_fields for i in field_to_idxs[fk]]
    val_idx   = [i for fk in val_fields   for i in field_to_idxs[fk]]
    return train_idx, val_idx


# ----------------------------
# Cache split so all models use identical indices
# ----------------------------
def folder_fingerprint(folder: str) -> str:
    items = []
    for fn in sorted(os.listdir(folder)):
        if not fn.lower().endswith(".tif"):
            continue
        p = os.path.join(folder, fn)
        st = os.stat(p)
        items.append(f"{fn}|{st.st_size}|{int(st.st_mtime)}")
    return hashlib.md5("\n".join(items).encode("utf-8")).hexdigest()

def load_or_make_split(folder: str, cache_path: str, seed=42, val_frac_each_group=0.20):
    fp = folder_fingerprint(folder)
    if os.path.exists(cache_path):
        with open(cache_path, "rb") as f:
            cache = pickle.load(f)
        if cache.get("fingerprint") == fp and cache.get("seed")==seed and cache.get("val_frac")==val_frac_each_group:
            print(f"[SPLIT CACHE] loaded {cache_path}")
            return cache["samples"], cache["train_idx"], cache["val_idx"]

    samples = scan_folder(folder)
    train_idx, val_idx = make_field_split(samples, val_frac_each_group=val_frac_each_group, seed=seed)
    os.makedirs(os.path.dirname(cache_path), exist_ok=True)
    with open(cache_path, "wb") as f:
        pickle.dump(
            dict(fingerprint=fp, seed=seed, val_frac=val_frac_each_group,
                 samples=samples, train_idx=train_idx, val_idx=val_idx),
            f, protocol=pickle.HIGHEST_PROTOCOL
        )
    print(f"[SPLIT CACHE] saved {cache_path}")
    return samples, train_idx, val_idx


# ----------------------------
# Dataset: same preprocessing used in training earlier
# (mask from band1 & band6 intersection, robust med/MAD norm on valid, remask)
# ----------------------------
class TifDatasetLikeTraining(Dataset):
    def __init__(self, samples: List[SampleMeta], indices: List[int], normalize=True):
        self.samples = samples
        self.indices = indices
        self.normalize = normalize
        self.eps = 1e-6
        self.b1 = 0
        self.b6 = 5

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, k):
        meta = self.samples[self.indices[k]]
        with rasterio.open(meta.path) as src:
            arr = src.read().astype(np.float32)  # [10,H,W]
        x = torch.from_numpy(arr)

        b1 = x[self.b1]; b6 = x[self.b6]
        valid = torch.isfinite(b1) & torch.isfinite(b6) & (b1 != 0) & (b6 != 0)

        x = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

        if self.normalize and valid.any():
            for b in range(x.shape[0]):
                vals = x[b][valid]
                if vals.numel() < 10:
                    continue
                med = vals.median()
                mad = (vals - med).abs().median()
                scale = 1.4826 * mad
                if (not torch.isfinite(scale)) or (scale < self.eps):
                    continue
                x[b] = (x[b] - med) / (scale + self.eps)

        x[:, ~valid] = 0.0
        y = torch.tensor(float(meta.cl), dtype=torch.float32)
        return x, y, valid


def collate_pad(batch):
    xs, ys, vms = zip(*batch)
    B = len(xs); C = xs[0].shape[0]
    Hmax = max(x.shape[1] for x in xs)
    Wmax = max(x.shape[2] for x in xs)

    x_pad = torch.zeros((B, C, Hmax, Wmax), dtype=torch.float32)
    vm_pad = torch.zeros((B, Hmax, Wmax), dtype=torch.bool)
    y = torch.stack(list(ys), dim=0)  # [B]

    for b in range(B):
        x = xs[b]; vm = vms[b]
        H, W = x.shape[1], x.shape[2]
        x_pad[b, :, :H, :W] = x
        vm_pad[b, :H, :W] = vm

    return x_pad, y, vm_pad


# ----------------------------
# Balanced batch sampler 50/50
# ----------------------------
class BalancedBatchSampler(Sampler[List[int]]):
    def __init__(self, dataset: Dataset, batch_size: int, seed: int = 0):
        assert batch_size % 2 == 0
        self.dataset = dataset
        self.batch_size = batch_size
        self.half = batch_size // 2
        self.rng = random.Random(seed)

        pos, neg = [], []
        for di in range(len(dataset)):
            _, y, _ = dataset[di]
            (pos if int(y.item()) == 1 else neg).append(di)
        if not pos or not neg:
            raise RuntimeError(f"Need both classes. pos={len(pos)} neg={len(neg)}")

        self.pos = pos
        self.neg = neg

    def __iter__(self):
        n_batches = math.ceil(max(len(self.pos), len(self.neg)) / self.half)
        pos_pool, neg_pool = self.pos[:], self.neg[:]
        self.rng.shuffle(pos_pool); self.rng.shuffle(neg_pool)

        def take(pool, src, k):
            out = []
            while len(out) < k:
                if not pool:
                    pool.extend(src[:])
                    self.rng.shuffle(pool)
                out.append(pool.pop())
            return out

        for _ in range(n_batches):
            batch = take(pos_pool, self.pos, self.half) + take(neg_pool, self.neg, self.half)
            self.rng.shuffle(batch)
            yield batch

    def __len__(self):
        return math.ceil(max(len(self.pos), len(self.neg)) / self.half)


# ----------------------------
# Metrics & evaluation (thresholds 0.5/0.75/0.95)
# ----------------------------
def confusion_from_probs(probs: np.ndarray, y: np.ndarray, thr: float):
    pred = (probs >= thr).astype(np.int32)
    y = y.astype(np.int32)
    tp = int(((pred==1)&(y==1)).sum())
    tn = int(((pred==0)&(y==0)).sum())
    fp = int(((pred==1)&(y==0)).sum())
    fn = int(((pred==0)&(y==1)).sum())
    return tp, tn, fp, fn

def metrics(tp, tn, fp, fn):
    n = tp+tn+fp+fn
    UA = tp / max(1, tp+fp)               # precision for class1
    PA = tp / max(1, tp+fn)               # recall for class1
    OA = (tp+tn) / max(1, n)
    PA_neg = tn / max(1, tn+fp)
    AA = 0.5*(PA + PA_neg)                # balanced accuracy
    F1 = 2*UA*PA / max(1e-12, UA+PA)
    return dict(UA=UA, PA=PA, OA=OA, AA=AA, F1=F1, N=n)

@torch.no_grad()
def predict_probs(model, loader, device="cuda", use_amp=True):
    model.eval()
    probs_all, y_all = [], []
    for x, y, vm in tqdm(loader, desc="VAL inference", leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        vm = vm.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=(use_amp and device=="cuda")):
            logits = model(x, valid_mask=vm)
        probs = torch.sigmoid(logits).float().cpu().numpy()
        probs_all.append(probs)
        y_all.append(y.float().cpu().numpy())
    return np.concatenate(probs_all), np.concatenate(y_all)

def full_validation_report(model, val_loader, device="cuda", thresholds=(0.5,0.75,0.95), use_amp=True):
    probs, y = predict_probs(model, val_loader, device=device, use_amp=use_amp)
    report = {}
    for thr in thresholds:
        tp, tn, fp, fn = confusion_from_probs(probs, y, thr)
        report[thr] = dict(tp=tp, tn=tn, fp=fp, fn=fn, **metrics(tp,tn,fp,fn))
    return report


# ----------------------------
# Shared training loop (5 epochs; LR drop at 3,4,5 => milestones 2,3,4)
# ----------------------------
def train_model(
    model: nn.Module,
    train_loader,
    val_loader,
    device=None,
    epochs=5,
    lr=3e-4,
    weight_decay=1e-4,
    use_amp=True,
    val_every=50,
    log_every=10,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[4,6,8], gamma=0.2)
    scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device=="cuda"))
    criterion = nn.BCEWithLogitsLoss()

    global_step = 0
    for ep in range(1, epochs+1):
        model.train()
        running_loss = 0.0
        running_corr = 0
        running_n = 0
        t0 = time.time()

        pbar = tqdm(enumerate(train_loader, start=1), total=len(train_loader), desc=f"TRAIN epoch {ep}/{epochs}")
        val_iter = iter(val_loader)

        for bi, (x,y,vm) in pbar:
            global_step += 1
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            vm = vm.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda", enabled=(use_amp and device=="cuda")):
                logits = model(x, valid_mask=vm)
                loss = criterion(logits, y)

            if not torch.isfinite(loss):
                print(f"⚠️ non-finite loss at step={global_step}, skipping")
                continue

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()

            probs = torch.sigmoid(logits.detach())
            pred = (probs >= 0.5).long()
            yt = y.long()
            corr = int((pred==yt).sum().item())
            n = int(yt.numel())

            running_loss += float(loss.item()) * x.size(0)
            running_corr += corr
            running_n += n

            avg_loss = running_loss / max(1, running_n)
            avg_acc = running_corr / max(1, running_n)
            lr_now = opt.param_groups[0]["lr"]

            if (global_step % val_every) == 0:
                # quick val batch only
                try:
                    vx, vy, vvm = next(val_iter)
                except StopIteration:
                    val_iter = iter(val_loader)
                    vx, vy, vvm = next(val_iter)
                vx = vx.to(device); vy = vy.to(device); vvm = vvm.to(device)
                with torch.amp.autocast("cuda", enabled=(use_amp and device=="cuda")):
                    vlogits = model(vx, valid_mask=vvm)
                    vloss = criterion(vlogits, vy).item()
                    vacc = ((torch.sigmoid(vlogits)>=0.5).long() == vy.long()).float().mean().item()
                pbar.set_postfix({"lr":f"{lr_now:.2e}", "loss":f"{loss.item():.4f}", "acc":f"{corr/n:.3f}",
                                  "avg_loss":f"{avg_loss:.4f}", "avg_acc":f"{avg_acc:.3f}",
                                  "val_b_loss":f"{vloss:.4f}", "val_b_acc":f"{vacc:.3f}"})
            else:
                pbar.set_postfix({"lr":f"{lr_now:.2e}", "loss":f"{loss.item():.4f}", "acc":f"{corr/n:.3f}",
                                  "avg_loss":f"{avg_loss:.4f}", "avg_acc":f"{avg_acc:.3f}"})

            if (bi % log_every) == 0:
                print(f"[ep{ep} b{bi:04d} step{global_step}] loss={loss.item():.4f} avg_loss={avg_loss:.4f} avg_acc={avg_acc:.3f} lr={lr_now:.2e} elapsed={time.time()-t0:.1f}s")

        sched.step()
        print(f"\n=== Epoch {ep} complete | TRAIN avg_loss={avg_loss:.4f} avg_acc={avg_acc:.4f} | lr now {opt.param_groups[0]['lr']:.2e}")

        rep = full_validation_report(model, val_loader, device=device, thresholds=(0.5,0.75,0.95), use_amp=use_amp)
        print("=== FULL VAL REPORT ===")
        for thr, r in rep.items():
            print(f"thr={thr}: TP={r['tp']} TN={r['tn']} FP={r['fp']} FN={r['fn']} | "
                  f"UA={r['UA']:.3f} PA={r['PA']:.3f} OA={r['OA']:.3f} AA={r['AA']:.3f} F1={r['F1']:.3f}")

    return model

In [ ]:
DATA_FOLDER = r"/tr_data_cutted3"
SPLIT_CACHE = os.path.join(DATA_FOLDER, "_cache", "split_val20_seed42.pkl")

samples, train_idx, val_idx = load_or_make_split(DATA_FOLDER, SPLIT_CACHE, seed=42, val_frac_each_group=0.20)
print("train images:", len(train_idx), "val images:", len(val_idx))

train_ds = TifDatasetLikeTraining(samples, train_idx, normalize=True)
val_ds   = TifDatasetLikeTraining(samples, val_idx,   normalize=True)

BATCH_SIZE = 8
NUM_WORKERS = 4

train_loader = DataLoader(
    train_ds,
    batch_sampler=BalancedBatchSampler(train_ds, batch_size=BATCH_SIZE, seed=42),
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_pad,
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_pad,
)

print("Loaders ready ✅  (same split for all models)")

# Models

In [ ]:
#Common CNN
import torch.nn.functional as F

def resize_mask_like(mask: torch.Tensor, feat: torch.Tensor) -> torch.Tensor:
    if mask is None:
        return None
    Hf, Wf = feat.shape[-2], feat.shape[-1]
    m = mask.unsqueeze(1).float()
    m = F.interpolate(m, size=(Hf, Wf), mode="nearest")
    return (m.squeeze(1) > 0)

def safe_maxpool2d(x: torch.Tensor, k: int = 2, s: int = 2) -> torch.Tensor:
    """
    Downsample only if both spatial dims are >= k.
    Otherwise return x unchanged.
    """
    H, W = x.shape[-2], x.shape[-1]
    if H < k or W < k:
        return x
    return F.max_pool2d(x, kernel_size=k, stride=s)

def safe_pool_mask(mask: torch.Tensor, x_after_pool: torch.Tensor) -> torch.Tensor:
    """
    Simply resize mask to match current feature map (works even if pooling skipped).
    """
    return resize_mask_like(mask, x_after_pool)

def masked_global_avg_pool(feat: torch.Tensor, mask: torch.Tensor):
    m = mask.unsqueeze(1).float()
    denom = m.sum(dim=(2,3)).clamp(min=1.0)
    return (feat * m).sum(dim=(2,3)) / denom


class FastCNNClassifier(nn.Module):
    def __init__(self, in_ch=10, base=64):
        super().__init__()
        self.b1 = nn.Sequential(
            nn.Conv2d(in_ch, base, 3, padding=1),
            nn.BatchNorm2d(base),
            nn.GELU(),
            nn.Conv2d(base, base, 3, padding=1),
            nn.BatchNorm2d(base),
            nn.GELU(),
        )
        self.b2 = nn.Sequential(
            nn.Conv2d(base, base*2, 3, padding=1),
            nn.BatchNorm2d(base*2),
            nn.GELU(),
            nn.Conv2d(base*2, base*2, 3, padding=1),
            nn.BatchNorm2d(base*2),
            nn.GELU(),
        )
        self.b3 = nn.Sequential(
            nn.Conv2d(base*2, base*4, 3, padding=1),
            nn.BatchNorm2d(base*4),
            nn.GELU(),
            nn.Conv2d(base*4, base*4, 3, padding=1),
            nn.BatchNorm2d(base*4),
            nn.GELU(),
        )
        self.head = nn.Sequential(
            nn.Linear(base*4, base*2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(base*2, 1)
        )

    def forward(self, x, valid_mask=None):
        m = valid_mask  # [B,H,W]

        x = self.b1(x)
        m = resize_mask_like(m, x)

        x = safe_maxpool2d(x, 2, 2)
        m = safe_pool_mask(m, x)

        x = self.b2(x)
        m = resize_mask_like(m, x)

        x = safe_maxpool2d(x, 2, 2)  # <-- will be skipped if H<2 or W<2
        m = safe_pool_mask(m, x)

        x = self.b3(x)
        m = resize_mask_like(m, x)

        pooled = masked_global_avg_pool(x, m) if m is not None else x.mean(dim=(2,3))
        return self.head(pooled).squeeze(1)

cnn = FastCNNClassifier(in_ch=10, base=64)
cnn = train_model(cnn, train_loader, val_loader, epochs=10, lr=1e-3, weight_decay=1e-4, val_every=10000, log_every=10000)
torch.save(cnn.state_dict(), "cnn_final_10epoch_v2.pth")


In [ ]:
#ConvLSTM
class ConvLSTMCell(nn.Module):
    def __init__(self, in_ch, hid_ch, k=3):
        super().__init__()
        p = k//2
        self.conv = nn.Conv2d(in_ch + hid_ch, 4*hid_ch, k, padding=p)

    def forward(self, x, h, c):
        # x [B,in,H,W], h/c [B,hid,H,W]
        combined = torch.cat([x, h], dim=1)
        gates = self.conv(combined)
        i,f,o,g = torch.chunk(gates, 4, dim=1)
        i = torch.sigmoid(i); f = torch.sigmoid(f); o = torch.sigmoid(o); g = torch.tanh(g)
        c = f*c + i*g
        h = o*torch.tanh(c)
        return h,c

class ConvLSTMClassifier(nn.Module):
    def __init__(self, in_ch=5, hid=64):
        super().__init__()
        self.cell = ConvLSTMCell(in_ch, hid, k=3)
        self.proj = nn.Sequential(
            nn.Conv2d(hid, hid, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(hid, hid, 3, padding=1),
            nn.GELU()
        )
        self.head = nn.Sequential(
            nn.Linear(hid, hid//2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hid//2, 1),
        )

    def forward(self, x, valid_mask=None):
        # split into T=2, each 5 bands
        x1 = x[:, :5, :, :]
        x2 = x[:, 5:, :, :]
        B, _, H, W = x1.shape
        h = torch.zeros((B, 64, H, W), device=x.device, dtype=x.dtype)
        c = torch.zeros((B, 64, H, W), device=x.device, dtype=x.dtype)
        h,c = self.cell(x1, h, c)
        h,c = self.cell(x2, h, c)
        feat = self.proj(h)

        pooled = masked_global_avg_pool(feat, valid_mask) if valid_mask is not None else feat.mean(dim=(2,3))
        return self.head(pooled).squeeze(1)

convlstm = ConvLSTMClassifier(in_ch=5, hid=64)
convlstm = train_model(convlstm, train_loader, val_loader, epochs=10, lr=1e-3, weight_decay=1e-4, val_every=10000, log_every=10000)
torch.save(convlstm.state_dict(), "convlstm_final_10epoch.pth")

In [ ]:
#ConvLSTM with attention
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---- reuse your masked_global_avg_pool from Cell 2 ----
def masked_global_avg_pool(feat: torch.Tensor, mask: torch.Tensor):
    # feat [B,C,H,W], mask [B,H,W] bool
    m = mask.unsqueeze(1).float()
    denom = m.sum(dim=(2,3)).clamp(min=1.0)
    pooled = (feat * m).sum(dim=(2,3)) / denom
    return pooled


# ----------------------------
# ConvLSTM Cell (same as before)
# ----------------------------
class ConvLSTMCell(nn.Module):
    def __init__(self, in_ch, hid_ch, k=3):
        super().__init__()
        p = k // 2
        self.conv = nn.Conv2d(in_ch + hid_ch, 4 * hid_ch, k, padding=p)

    def forward(self, x, h, c):
        combined = torch.cat([x, h], dim=1)
        gates = self.conv(combined)
        i, f, o, g = torch.chunk(gates, 4, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)
        c = f * c + i * g
        h = o * torch.tanh(c)
        return h, c


# ----------------------------
# Spatial Attention blocks (CBAM-ish)
# ----------------------------
class ChannelAttention(nn.Module):
    """
    Channel attention using avg/max pooling -> shared MLP (implemented with 1x1 convs).
    """
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        hidden = max(8, channels // reduction)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, hidden, kernel_size=1, bias=False),
            nn.GELU(),
            nn.Conv2d(hidden, channels, kernel_size=1, bias=False),
        )

    def forward(self, x):
        # x [B,C,H,W]
        avg = F.adaptive_avg_pool2d(x, 1)
        mx  = F.adaptive_max_pool2d(x, 1)
        w = torch.sigmoid(self.mlp(avg) + self.mlp(mx))
        return x * w


class SpatialAttention(nn.Module):
    """
    Spatial attention using channel-wise avg/max -> conv -> sigmoid.
    """
    def __init__(self, k: int = 7):
        super().__init__()
        p = k // 2
        self.conv = nn.Conv2d(2, 1, kernel_size=k, padding=p, bias=False)

    def forward(self, x):
        # x [B,C,H,W]
        avg = x.mean(dim=1, keepdim=True)
        mx, _ = x.max(dim=1, keepdim=True)
        a = torch.cat([avg, mx], dim=1)  # [B,2,H,W]
        w = torch.sigmoid(self.conv(a))  # [B,1,H,W]
        return x * w


class CBAM(nn.Module):
    def __init__(self, channels: int, reduction: int = 16, spatial_k: int = 7):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction=reduction)
        self.sa = SpatialAttention(k=spatial_k)

    def forward(self, x):
        x = self.ca(x)
        x = self.sa(x)
        return x


# ----------------------------
# ConvLSTM + spatial attention classifier
# ----------------------------
class ConvLSTMSpatialAttnClassifier(nn.Module):
    """
    - Takes x [B,10,H,W]
    - Splits into 2 steps: x1=[B,5,H,W], x2=[B,5,H,W]
    - ConvLSTM over time (2 steps)
    - Spatial attention (CBAM) on final hidden map (focus on damaged spatial regions)
    - Masked pooling using valid_mask
    """
    def __init__(self, in_ch=5, hid=64, attn_reduction=16, spatial_k=7):
        super().__init__()
        self.hid = hid
        self.cell = ConvLSTMCell(in_ch, hid, k=3)

        self.refine = nn.Sequential(
            nn.Conv2d(hid, hid, 3, padding=1, bias=False),
            nn.BatchNorm2d(hid),
            nn.GELU(),
            nn.Conv2d(hid, hid, 3, padding=1, bias=False),
            nn.BatchNorm2d(hid),
            nn.GELU(),
        )

        self.cbam = CBAM(hid, reduction=attn_reduction, spatial_k=spatial_k)

        self.head = nn.Sequential(
            nn.Linear(hid, hid // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hid // 2, 1),
        )

    def forward(self, x, valid_mask=None):
        x1 = x[:, :5, :, :]
        x2 = x[:, 5:, :, :]

        B, _, H, W = x1.shape
        h = torch.zeros((B, self.hid, H, W), device=x.device, dtype=x.dtype)
        c = torch.zeros((B, self.hid, H, W), device=x.device, dtype=x.dtype)

        h, c = self.cell(x1, h, c)
        h, c = self.cell(x2, h, c)

        feat = self.refine(h)
        feat = self.cbam(feat)   # <-- spatial attention focus

        pooled = masked_global_avg_pool(feat, valid_mask) if valid_mask is not None else feat.mean(dim=(2, 3))
        return self.head(pooled).squeeze(1)


# ---- train it using the SAME train_model + SAME loaders ----
convlstm_spatial = ConvLSTMSpatialAttnClassifier(in_ch=5, hid=64, attn_reduction=16, spatial_k=7)
convlstm_spatial = train_model(
    convlstm_spatial,
    train_loader,
    val_loader,
    epochs=10,
    lr=1e-3,
    weight_decay=1e-4,
    val_every=5000,
    log_every=5000,
)
torch.save(convlstm_spatial.state_dict(), "convlstm_spatial_final_10epoch.pth")

In [ ]:
#Basic Trasnformer
import torch
import torch.nn as nn
import torch.nn.functional as F


def pad_to_min_hw(x: torch.Tensor, min_h: int, min_w: int):
    """
    Pads x [B,C,H,W] with zeros on bottom/right to ensure H>=min_h and W>=min_w.
    """
    H, W = x.shape[-2], x.shape[-1]
    pad_h = max(0, min_h - H)
    pad_w = max(0, min_w - W)
    if pad_h > 0 or pad_w > 0:
        x = F.pad(x, (0, pad_w, 0, pad_h))  # (left,right,top,bottom)
    return x
    
def resize_mask_like(mask: torch.Tensor, feat: torch.Tensor) -> torch.Tensor:
    if mask is None:
        return None
    Hf, Wf = feat.shape[-2], feat.shape[-1]
    m = mask.unsqueeze(1).float()
    m = F.interpolate(m, size=(Hf, Wf), mode="nearest")
    return (m.squeeze(1) > 0)

def masked_global_avg_pool(feat: torch.Tensor, mask: torch.Tensor):
    m = mask.unsqueeze(1).float()
    denom = m.sum(dim=(2,3)).clamp(min=1.0)
    return (feat * m).sum(dim=(2,3)) / denom


class ConvMixerBlock(nn.Module):
    """
    Token mixing WITHOUT attention:
      - depthwise conv mixes spatial tokens (works for any H,W)
    Channel mixing:
      - pointwise MLP via 1x1 convs
    """
    def __init__(self, dim: int, k: int = 5, drop: float = 0.1):
        super().__init__()
        p = k // 2
        self.token_mix = nn.Sequential(
            nn.Conv2d(dim, dim, kernel_size=k, padding=p, groups=dim, bias=False),  # depthwise
            nn.BatchNorm2d(dim),
            nn.GELU(),
        )
        self.channel_mix = nn.Sequential(
            nn.Conv2d(dim, 4*dim, kernel_size=1, bias=False),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Conv2d(4*dim, dim, kernel_size=1, bias=False),
            nn.Dropout(drop),
        )
        self.norm = nn.BatchNorm2d(dim)

    def forward(self, x):
        # x: [B,dim,H,W]
        x = x + self.token_mix(x)
        x = x + self.channel_mix(self.norm(x))
        return x


class BasicConvMixerClassifier(nn.Module):
    def __init__(self, in_ch=10, patch=4, dim=128, depth=8, token_kernel=5, drop=0.1):
        super().__init__()
        self.patch = patch
        self.embed = nn.Sequential(
            nn.Conv2d(in_ch, dim, kernel_size=patch, stride=patch, bias=False),
            nn.BatchNorm2d(dim),
            nn.GELU(),
        )
        self.blocks = nn.ModuleList([
            ConvMixerBlock(dim=dim, k=token_kernel, drop=drop) for _ in range(depth)
        ])
        self.head = nn.Sequential(
            nn.Linear(dim, dim//2),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(dim//2, 1)
        )

    def forward(self, x, valid_mask=None):
        # ✅ ensure patch embed always works even for tiny chips (H<patch or W<patch)
        x = pad_to_min_hw(x, self.patch, self.patch)

        feat = self.embed(x)  # [B,dim,Hp,Wp]
        for blk in self.blocks:
            feat = blk(feat)

        if valid_mask is not None:
            # mask must be padded the same way as x so alignment is preserved
            vm = valid_mask
            # pad mask to same min size (bool -> float pad -> back to bool)
            vm = pad_to_min_hw(vm.unsqueeze(1).float(), self.patch, self.patch).squeeze(1) > 0
            m = resize_mask_like(vm, feat)
            pooled = masked_global_avg_pool(feat, m)
        else:
            pooled = feat.mean(dim=(2,3))

        return self.head(pooled).squeeze(1)

basic_no_attn = BasicConvMixerClassifier(
    in_ch=10,
    patch=4,
    dim=128,
    depth=8,
    token_kernel=5,
    drop=0.1,
)

basic_no_attn = train_model(
    basic_no_attn,
    train_loader,
    val_loader,
    epochs=10,
    lr=1e-4,
    weight_decay=1e-4,
    val_every=10000,
    log_every=10000,
)
torch.save(basic_no_attn.state_dict(), "basic_transformer_final_10epoch.pth")


In [ ]:
# Window Attention
def pad_to_min_hw(x: torch.Tensor, min_h: int, min_w: int):
    """
    Pads x [B,C,H,W] with zeros on bottom/right to ensure H>=min_h and W>=min_w.
    Returns padded x and original (H,W).
    """
    H, W = x.shape[-2], x.shape[-1]
    pad_h = max(0, min_h - H)
    pad_w = max(0, min_w - W)
    if pad_h > 0 or pad_w > 0:
        x = F.pad(x, (0, pad_w, 0, pad_h))  # (left,right,top,bottom)
    return x, (H, W)
def window_partition(x, ws):
    # x [B,H,W,C] -> [B*nW,ws,ws,C]
    B,H,W,C = x.shape
    x = x.view(B, H//ws, ws, W//ws, ws, C)
    return x.permute(0,1,3,2,4,5).contiguous().view(-1, ws, ws, C)

def window_reverse(w, ws, H, W, B):
    # w [B*nW,ws,ws,C] -> [B,H,W,C]
    C = w.shape[-1]
    x = w.view(B, H//ws, W//ws, ws, ws, C)
    return x.permute(0,1,3,2,4,5).contiguous().view(B,H,W,C)

class WindowAttention(nn.Module):
    def __init__(self, dim, heads, drop=0.0):
        super().__init__()
        self.heads = heads
        self.dim = dim
        self.hd = dim // heads
        assert dim % heads == 0
        self.qkv = nn.Linear(dim, 3*dim)
        self.proj = nn.Linear(dim, dim)
        self.drop = drop

    def forward(self, x):
        # x [Bwin,N,C]
        Bwin,N,C = x.shape
        qkv = self.qkv(x).reshape(Bwin,N,3,self.heads,self.hd).permute(2,0,3,1,4)
        q,k,v = qkv[0], qkv[1], qkv[2]
        out = F.scaled_dot_product_attention(q,k,v, dropout_p=self.drop if self.training else 0.0)
        out = out.transpose(1,2).reshape(Bwin,N,C)
        return self.proj(out)

class SwinLiteBlock(nn.Module):
    def __init__(self, dim, heads, ws=8, shift=0, drop=0.1):
        super().__init__()
        self.ws = ws
        self.shift = shift
        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(dim, heads, drop=drop)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, 4*dim), nn.GELU(), nn.Dropout(drop),
            nn.Linear(4*dim, dim), nn.Dropout(drop)
        )

    def forward(self, x, H, W):
        B,L,C = x.shape
        x0 = x
        x = self.norm1(x).view(B,H,W,C)
        pad_h = (self.ws - (H % self.ws)) % self.ws
        pad_w = (self.ws - (W % self.ws)) % self.ws
        if pad_h or pad_w:
            x = F.pad(x, (0,0, 0,pad_w, 0,pad_h))
        Hp,Wp = x.shape[1], x.shape[2]

        if self.shift > 0:
            x = torch.roll(x, shifts=(-self.shift,-self.shift), dims=(1,2))

        w = window_partition(x, self.ws).view(-1, self.ws*self.ws, C)
        w = self.attn(w)
        w = w.view(-1, self.ws, self.ws, C)
        x = window_reverse(w, self.ws, Hp, Wp, B)

        if self.shift > 0:
            x = torch.roll(x, shifts=(self.shift,self.shift), dims=(1,2))

        x = x[:, :H, :W, :].reshape(B, H*W, C)
        x = x0 + x
        x = x + self.mlp(self.norm2(x))
        return x

class WindowTransformerClassifier(nn.Module):
    def __init__(self, in_ch=10, dim=128, depth=8, heads=4, ws=8, drop=0.1):
        super().__init__()
        self.patch = nn.Conv2d(in_ch, dim, kernel_size=4, stride=4)
        blocks = []
        for i in range(depth):
            shift = 0 if (i%2==0) else ws//2
            blocks.append(SwinLiteBlock(dim, heads, ws=ws, shift=shift, drop=drop))
        self.blocks = nn.ModuleList(blocks)
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Sequential(
            nn.Linear(dim, dim//2), nn.GELU(), nn.Dropout(drop),
            nn.Linear(dim//2, 1)
        )

    def forward(self, x, valid_mask=None):
        # 1) pad tiny chips so patch embed conv (k=4) always works
        x, _ = pad_to_min_hw(x, min_h=4, min_w=4)
    
        # 2) patch embed
        feat = self.patch(x)  # [B,C,Hp,Wp]
        B, C, Hp, Wp = feat.shape
        t = feat.permute(0, 2, 3, 1).reshape(B, Hp * Wp, C)
    
        # 3) transformer blocks
        for blk in self.blocks:
            t = blk(t, Hp, Wp)
        t = self.norm(t)
    
        # 4) masked pooling (resize mask to Hp,Wp instead of pooling)
        if valid_mask is not None:
            m = resize_mask_like(valid_mask, feat)      # [B,Hp,Wp]
            keep = m.view(B, -1).float()
            denom = keep.sum(dim=1).clamp(min=1.0).unsqueeze(-1)
            pooled = (t * keep.unsqueeze(-1)).sum(dim=1) / denom
        else:
            pooled = t.mean(dim=1)
    
        return self.head(pooled).squeeze(1)

wt = WindowTransformerClassifier(in_ch=10, dim=128, depth=8, heads=4, ws=8, drop=0.1)
wt = train_model(wt, train_loader, val_loader, epochs=10, lr=1e-4, weight_decay=1e-4, val_every=10000, log_every=10000)
torch.save(wt.state_dict(), "window_attention_transformer_final_10epoch.pth")

In [ ]:
#Dual Swin Tranformer

import torch
import torch.nn.functional as F
import torch.nn.functional as F

def pad_to_min_hw(x, min_h: int, min_w: int):
    H, W = x.shape[-2], x.shape[-1]
    pad_h = max(0, min_h - H)
    pad_w = max(0, min_w - W)
    if pad_h > 0 or pad_w > 0:
        x = F.pad(x, (0, pad_w, 0, pad_h))  # right, bottom
    return x
def resize_mask_like(mask, feat):
    if mask is None:
        return None
    Hf, Wf = feat.shape[-2], feat.shape[-1]
    m = mask.unsqueeze(1).float()
    m = F.interpolate(m, size=(Hf, Wf), mode="nearest")
    return (m.squeeze(1) > 0)
    
class DualSwinFusionClassifier(nn.Module):
    """
    Two-stream patch embed for damage (first 5 bands) and reference (last 5 bands),
    token fusion: [D, R, |D-R|, D*R] -> proj -> Swin blocks -> masked pooled -> classifier
    """
    def __init__(self, dim=160, depth=12, heads=5, ws=8, drop=0.1):
        super().__init__()
        self.patch_d = nn.Conv2d(5, dim, kernel_size=4, stride=4)
        self.patch_r = nn.Conv2d(5, dim, kernel_size=4, stride=4)
        self.fuse = nn.Sequential(
            nn.Linear(4*dim, dim),
            nn.GELU(),
            nn.Dropout(drop)
        )
        blocks = []
        for i in range(depth):
            shift = 0 if (i%2==0) else ws//2
            blocks.append(SwinLiteBlock(dim, heads, ws=ws, shift=shift, drop=drop))
        self.blocks = nn.ModuleList(blocks)
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Sequential(
            nn.Linear(dim, dim//2), nn.GELU(), nn.Dropout(drop),
            nn.Linear(dim//2, 1)
        )

    def forward(self, x, valid_mask=None):
        # ensure patch embedding (k=4) works for tiny chips
        x = pad_to_min_hw(x, 4, 4)
        if valid_mask is not None:
            valid_mask = pad_to_min_hw(valid_mask.unsqueeze(1).float(), 4, 4).squeeze(1) > 0
    
        d = x[:, :5, :, :]
        r = x[:, 5:, :, :]
    
        fd = self.patch_d(d)   # now safe
        fr = self.patch_r(r)
        B,C,Hp,Wp = fd.shape
        td = fd.permute(0,2,3,1).reshape(B, Hp*Wp, C)
        tr = fr.permute(0,2,3,1).reshape(B, Hp*Wp, C)
        t = torch.cat([td, tr, (td-tr).abs(), (td*tr)], dim=-1)
        t = self.fuse(t)

        for blk in self.blocks:
            t = blk(t, Hp, Wp)
        t = self.norm(t)

        if valid_mask is not None:
            m = resize_mask_like(valid_mask, fd)   # fd is [B,C,Hp,Wp]
            keep = m.view(B, -1).float()
            denom = keep.sum(dim=1).clamp(min=1.0).unsqueeze(-1)
            pooled = (t * keep.unsqueeze(-1)).sum(dim=1) / denom
        else:
            pooled = t.mean(dim=1)

        return self.head(pooled).squeeze(1)

dual = DualSwinFusionClassifier(dim=160, depth=12, heads=5, ws=8, drop=0.1)
dual = train_model(dual, train_loader, val_loader, epochs=10, lr=1e-4, weight_decay=1e-4, val_every=10000, log_every=10000)
torch.save(dual.state_dict(), "dual_final_10epoch.pth")